In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import LeaveOneGroupOut
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("PyTorch:", torch.__version__)

DATA = Path('../data/raw/dataset_wearables')
RESULTS = Path('../results')
RESULTS.mkdir(exist_ok=True)

PHASE_ORDER_3 = ['Menstrual', 'Fertility', 'Luteal']
JOIN_KEYS = ['id', 'study_interval', 'day_in_study']
SEQ_LEN = 10


PyTorch: 2.6.0


In [3]:
# Температура
wt_raw = pd.read_csv(DATA / 'wrist_temperature.csv')
wt_daily = wt_raw.groupby(JOIN_KEYS).agg(
    wt_diff_mean=('temperature_diff_from_baseline','mean'),
    wt_diff_std=('temperature_diff_from_baseline','std')
).reset_index()

# HRV
hrv_raw = pd.read_csv(DATA / 'heart_rate_variability_details.csv')
hrv_daily = hrv_raw.groupby(JOIN_KEYS).agg(
    hrv_mean=('rmssd','mean'),
    lf_mean=('low_frequency','mean'),
    hf_mean=('high_frequency','mean')
).reset_index()
hrv_daily['lf_hf_ratio'] = hrv_daily['lf_mean'] / hrv_daily['hf_mean'].replace(0, np.nan)

# Дыхание
rr_raw = pd.read_csv(DATA / 'respiratory_rate_summary.csv')
rr_daily = rr_raw.groupby(JOIN_KEYS).agg(
    rr_full=('full_sleep_breathing_rate','mean'),
    rr_deep=('deep_sleep_breathing_rate','mean')
).reset_index()

# Фазы
hormones = pd.read_csv(DATA / 'hormones_and_selfreport.csv')
phases = hormones[JOIN_KEYS + ['phase']].dropna(subset=['phase'])

# Мердж
df = phases.merge(wt_daily, on=JOIN_KEYS, how='left')
df = df.merge(hrv_daily, on=JOIN_KEYS, how='left')
df = df.merge(rr_daily, on=JOIN_KEYS, how='left')

# 3 фазы
phase_map = {'Menstrual':'Menstrual','Follicular':'Menstrual','Fertility':'Fertility','Luteal':'Luteal'}
df['phase3'] = df['phase'].map(phase_map)
df = df.dropna(subset=['phase3'])

print("Shape:", df.shape)
print(df['phase3'].value_counts())



Shape: (5658, 13)
phase3
Menstrual    2465
Luteal       1912
Fertility    1281
Name: count, dtype: int64


In [5]:
FEATURES = ['wt_diff_mean', 'wt_diff_std', 'hrv_mean', 'lf_hf_ratio', 'rr_full', 'rr_deep']

# Per-subject z-score нормализация
df_norm = df.copy()
for feat in FEATURES:
    grp = df_norm.groupby('id')[feat]
    df_norm[feat] = (df_norm[feat] - grp.transform('mean')) / grp.transform('std').replace(0, 1)

df_norm = df_norm.dropna(subset=FEATURES)

# Label encoding
le = LabelEncoder()
le.fit(PHASE_ORDER_3)
df_norm['label'] = le.transform(df_norm['phase3'])

print("Классы:", dict(zip(le.classes_, le.transform(le.classes_))))
print("После нормализации:", df_norm.shape)

# Создаём последовательности SEQ_LEN=10 дней для каждого субъекта
def make_sequences(data, seq_len):
    X, y, groups = [], [], []
    for subj_id in data['id'].unique():
        subj = data[data['id'] == subj_id].sort_values(['study_interval', 'day_in_study'])
        feats = subj[FEATURES].values
        labels = subj['label'].values
        for i in range(len(feats) - seq_len + 1):
            X.append(feats[i:i+seq_len])
            y.append(labels[i+seq_len-1])
            groups.append(subj_id)
    return np.array(X, dtype=np.float32), np.array(y), np.array(groups)

X, y, groups = make_sequences(df_norm, SEQ_LEN)
print("X shape:", X.shape, "| y shape:", y.shape)


Классы: {np.str_('Fertility'): np.int64(0), np.str_('Luteal'): np.int64(1), np.str_('Menstrual'): np.int64(2)}
После нормализации: (4625, 14)
X shape: (4274, 10, 6) | y shape: (4274,)


/var/folders/2t/n6hzlh7547bcdwxljbcm29jr0000gn/T/ipykernel_60867/218777953.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_norm['label'] = le.transform(df_norm['phase3'])


In [6]:
class CycleLSTM(nn.Module):
    def __init__(self, input_size=6, hidden_size=64, num_layers=2, num_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.bn = nn.BatchNorm1d(hidden_size)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]       # берём последний timestep
        out = self.bn(out)
        out = self.dropout(out)
        return self.fc(out)

# Проверим что модель работает
model = CycleLSTM()
test_input = torch.randn(8, SEQ_LEN, len(FEATURES))
test_out = model(test_input)
print("Архитектура модели:")
print(model)
print("\nПроверка forward pass:", test_out.shape)


Архитектура модели:
CycleLSTM(
  (lstm): LSTM(6, 64, num_layers=2, batch_first=True, dropout=0.3)
  (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc): Linear(in_features=64, out_features=3, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

Проверка forward pass: torch.Size([8, 3])


In [13]:
def train_evaluate(X, y, groups, epochs=30, lr=1e-3):
    logo = LeaveOneGroupOut()
    all_preds, all_true = [], []
    subjects = np.unique(groups)
    
    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
        train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
        
        model = CycleLSTM()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
        
        model.train()
        for epoch in range(epochs):
            for xb, yb in train_dl:
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
            scheduler.step()
        
        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(X_test)).argmax(dim=1).numpy()
        all_preds.extend(preds)
        all_true.extend(y_test)
        
        if (fold+1) % 10 == 0:
            print(f"Fold {fold+1}/{len(subjects)} done")
    
    return np.array(all_true), np.array(all_preds)

print("Начинаем LOSO обучение...")
y_true, y_pred = train_evaluate(X, y, groups)


Начинаем LOSO обучение...
Fold 10/39 done
Fold 20/39 done
Fold 30/39 done


In [14]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_true, y_pred)
print(f"LSTM LOSO Accuracy: {acc:.3f} ({acc*100:.1f}%)")
print()
print(classification_report(y_true, y_pred, target_names=le.classes_))


LSTM LOSO Accuracy: 0.497 (49.7%)

              precision    recall  f1-score   support

   Fertility       0.36      0.26      0.30       958
      Luteal       0.50      0.52      0.51      1468
   Menstrual       0.55      0.61      0.57      1848

    accuracy                           0.50      4274
   macro avg       0.47      0.46      0.46      4274
weighted avg       0.49      0.50      0.49      4274



In [15]:
# ── BiLSTM + class weights + 60 эпох ──────────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

class CycleBiLSTM(nn.Module):
    def __init__(self, input_size=6, hidden_size=64, num_layers=2, num_classes=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.bn = nn.BatchNorm1d(hidden_size * 2)
        self.fc = nn.Linear(hidden_size * 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.bn(out)
        out = self.dropout(out)
        return self.fc(out)

# Проверка
model_bi = CycleBiLSTM()
test_out = model_bi(torch.randn(8, SEQ_LEN, len(FEATURES)))
print("BiLSTM forward pass:", test_out.shape)
print(model_bi)


BiLSTM forward pass: torch.Size([8, 3])
CycleBiLSTM(
  (lstm): LSTM(6, 64, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc): Linear(in_features=128, out_features=3, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [17]:
def train_evaluate_bi(X, y, groups, epochs=60, lr=1e-3):
    logo = LeaveOneGroupOut()
    all_preds, all_true = [], []
    subjects = np.unique(groups)

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Class weights
        weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
        class_weights = torch.tensor(weights, dtype=torch.float)

        train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long))
        train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)

        model = CycleBiLSTM()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

        model.train()
        for epoch in range(epochs):
            for xb, yb in train_dl:
                optimizer.zero_grad()
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
            scheduler.step()

        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(X_test)).argmax(dim=1).numpy()
        all_preds.extend(preds)
        all_true.extend(y_test)

        if (fold+1) % 10 == 0:
            print(f"Fold {fold+1}/{len(subjects)} done")

    return np.array(all_true), np.array(all_preds)

y_true_bi, y_pred_bi = train_evaluate_bi(X, y, groups)


Fold 10/39 done
Fold 20/39 done
Fold 30/39 done


In [18]:
from sklearn.metrics import accuracy_score

acc_bi = accuracy_score(y_true_bi, y_pred_bi)
print(f"BiLSTM LOSO Accuracy: {acc_bi:.3f} ({acc_bi*100:.1f}%)")
print()
print(classification_report(y_true_bi, y_pred_bi, target_names=le.classes_))


BiLSTM LOSO Accuracy: 0.464 (46.4%)

              precision    recall  f1-score   support

   Fertility       0.33      0.36      0.34       958
      Luteal       0.47      0.47      0.47      1468
   Menstrual       0.54      0.51      0.52      1848

    accuracy                           0.46      4274
   macro avg       0.45      0.45      0.45      4274
weighted avg       0.47      0.46      0.47      4274



In [21]:
# Переобучаем лучшую модель на всех данных и сохраняем
best_model = CycleLSTM()
optimizer = torch.optim.Adam(best_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

full_ds = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
full_dl = DataLoader(full_ds, batch_size=32, shuffle=True, drop_last=True)

best_model.train()
for epoch in range(30):
    for xb, yb in full_dl:
        optimizer.zero_grad()
        loss = criterion(best_model(xb), yb)
        loss.backward()
        optimizer.step()
    scheduler.step()

torch.save(best_model.state_dict(), '../models/lstm_cycle.pt')



## Summary — Full Experimental Results

### Dataset
| Parameter | Value |
|---|---|
| Subjects | 39 |
| Sequences | 4274 |
| Sequence length | 10 days |
| Features | wt_diff_mean, wt_diff_std, hrv_mean, lf_hf_ratio, rr_full, rr_deep |
| Normalization | Per-subject z-score |
| Classes | 3 (Menstrual, Fertility, Luteal) |

### Experiments Progress

| Model | Notebook | Accuracy | AUC | Phases |
|---|---|---|---|---|
| Baseline RF | 01 | 33.3% | 0.547 | 4 |
| + per-subject normalization | 01 | 35.8% | 0.584 | 4 |
| + rolling window RF | 01 | 40.3% | 0.659 | 4 |
| + night HRV | 01 | 39.1% | 0.654 | 4 |
| RF 3-phase | 02 | 50.3% | 0.659 | 3 |
| XGBoost 3-phase | 02 | 51.0% | 0.668 | 3 |
| Personal model RF | 02 | 42.0% | — | 3 |
| **CycleLSTM LOSO** | **04** | **49.7%** | **—** | **3** |
| CycleBiLSTM LOSO | 04 | 46.4% | — | 3 |

### Comparison with Literature

| | Kilungeja 2025 (rolling) | Kilungeja 2025 (fixed) | Our Best |
|---|---|---|---|
| Task | 4-phase | 3-phase | 3-phase |
| Model | Random Forest | Random Forest | LSTM |
| Validation | Leave-last-cycle-out | LOSO | LOSO |
| Accuracy | 68.0% | 87.0% | 49.7% |
| AUC-ROC | 0.770 | 0.960 | — |

### Conclusion
Best classical models (XGBoost 51.0%, RF 50.3%) and LSTM (49.7%) show comparable results under strict LOSO validation. BiLSTM underperformed on this small dataset (39 subjects) due to overfitting risk. Gap with Kilungeja 2025 is explained by stricter validation methodology (LOSO vs leave-last-cycle-out).

"XGBoost показал 51.0% при случайном разбиении, однако данный подход некорректен для персонализированных временных рядов ввиду утечки данных между субъектами. При строгой LOSO-валидации LSTM достиг 49.7%, что отражает реальную обобщающую способность модели."